In [0]:
from pyspark.sql.functions import *
import dlt

In [0]:
catalog       = "logistics"
source_schema = "bronze"

In [0]:
def apply_silver_schema(df: DataFrame, exclusive_col_expr: Column) -> DataFrame:

  base_cols = [
    col("id").cast("string"),
    col("vendorid").cast("double").cast("int").alias("vendorid"),
    col("pickup_datetime").cast("timestamp"),
    col("dropoff_datetime").cast("timestamp"),
    col("passenger_count").cast("double").cast("int").alias("passenger_count"),
    col("trip_distance").cast("double"),
    col("ratecodeid").cast("double").cast("int").alias("ratecodeid"),
    col("store_and_fwd_flag").cast("string"),
    col("pulocationid").cast("double").cast("int").alias("pulocationid"),
    col("dolocationid").cast("double").cast("int").alias("dolocationid"),
    col("payment_type").cast("double").cast("int").alias("payment_type"),
    col("fare_amount").cast("double").cast("decimal(10,2)").alias("fare_amount"),
    col("extra").cast("double").cast("decimal(10,2)").alias("extra"),
    col("mta_tax").cast("double").cast("decimal(10,2)").alias("mta_tax"),
    col("tip_amount").cast("double").cast("decimal(10,2)").alias("tip_amount"),
    col("tolls_amount").cast("double").cast("decimal(10,2)").alias("tolls_amount"),
    col("improvement_surcharge").cast("double").cast("decimal(10,2)").alias("improvement_surcharge"),
    col("total_amount").cast("double").cast("decimal(10,2)").alias("total_amount"),
    col("congestion_surcharge").cast("double").cast("decimal(10,2)").alias("congestion_surcharge"),
    current_timestamp().alias("_ingestion_at"),
    col("date_partition").cast("string")
  ]

  return df.select(*base_cols, exclusive_col_expr)

In [0]:
quality_conditions = {
    "valid_total_amount":    "total_amount >= 0",
    "valid_passenger_count": "passenger_count <= 4 AND passenger_count > 0",
    "valid_trip_distance":   "trip_distance >= 0",
    "consistent_timestamps": "dropoff_datetime > pickup_datetime"
}

In [0]:
def apply_quarantine_logic():

    return [
        when(col("total_amount") < 0, "Invalid Total Amount"),
        when((col("passenger_count") > 4) | (col("passenger_count") <= 0), "Invalid Passenger Count"),
        when(col("trip_distance") < 0, "Invalid Trip Distance"),
        when(col("dropoff_datetime") <= col("pickup_datetime"), "Inconsistent Timestamps")
    ]

In [0]:
green_table_name  = "green_taxi"
green_source_path = f"{catalog}.{source_schema}.{green_table_name}"
green_column_expr = col("trip_type").cast("double").cast("int").alias("trip_type")

# Green Bronze View 
@dlt.view
def green_prepared_view():
    return apply_silver_schema(dlt.read_stream(green_source_path), green_column_expr)

# Green Silver table
@dlt.table(
        name=f"silver_{green_table_name}",
        comment="Green cabs silver table partitioned by date",
        partition_cols=["date_partition"],
        table_properties={
            "quality": "silver"
        }
)
@dlt.expect_all_or_drop(quality_conditions)
def silver_green():
    return dlt.read_stream("green_prepared_view")

# Quarantine Data
@dlt.table(
        name=f"quarantine_{green_table_name}",
        comment="Quarantine green cabs table partitioned by date",
        partition_cols=["date_partition"]
)
def quarantine_green():
    return (
        dlt.read_stream("green_prepared_view")
        .withColumn("quarantine_reason", 
            concat_ws(" | ", *apply_quarantine_logic())
        )
        .filter(col("quarantine_reason") != "")
    )

In [0]:
yellow_table_name  = "yellow_taxi"
yellow_source_path = f"{catalog}.{source_schema}.{yellow_table_name}"
yellow_column_expr = col("airport_fee").cast("double").cast("decimal(10,2)").alias("airport_fee")

# Yellow Bronze View 
@dlt.view
def yellow_prepared_view():
    return apply_silver_schema(dlt.read_stream(yellow_source_path), yellow_column_expr)

# Yellow Silver table
@dlt.table(
        name=f"silver_{yellow_table_name}",
        comment="Yellow cabs silver table partitioned by date",
        partition_cols=["date_partition"],
        table_properties={
            "quality": "silver"
        }
)
@dlt.expect_all_or_drop(quality_conditions)
def silver_yellow():
    return dlt.read_stream("yellow_prepared_view")

# Quarantine Data
@dlt.table(
        name=f"quarantine_{yellow_table_name}",
        comment="Quarantine yellow cabs table partitioned by date",
        partition_cols=["date_partition"]
)
def quarantine_yellow():
    return (
        dlt.read_stream("yellow_prepared_view")
        .withColumn("quarantine_reason", 
            concat_ws(" | ", *apply_quarantine_logic())
        )
        .filter(col("quarantine_reason") != "")
    )